# 🛡️ Track 2: UniShield DFD Training — [ALL_REMAINING_FOLDS] (Folds 5–10)
### UniShield: Unified Forgery Image Detection & Localization (CVPR Findings 2026)

**⚡ PARALLEL HORIZONTAL CLOUD SHARD:** `ALL_REMAINING_FOLDS`
- **Assigned Folds for this Notebook Instance:** `[5, 6, 9, 10]`
- **Pre-seeded Completed Folds:**
  - Fold 1 (`ACC=99.63%`, `AUC=99.95%`)
  - Fold 2 (`ACC=99.83%`, `AUC=99.99%`)
  - Fold 7 (`ACC=99.83%`, `AUC=100.00%`)
- **Zero Identity Leakage:** Grouped by Video/Subject ID with fixed seed `42`.
- **Memory Safe:** Bounded chunk cache, persistent_workers disabled, explicit per-fold HDF5 handle release.


## 📌 Step 1: Dependencies & System Verification

In [ ]:
import os
import gc
import re
import time
import json
import hashlib
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import h5py
    print(f"✅ h5py {h5py.__version__} is ready.")
except ImportError:
    !pip install -q h5py
    import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"🚀 Active Compute Device: {device} (Detected {gpu_count} GPU(s))")
if torch.cuda.is_available():
    for g in range(gpu_count):
        print(f"   • GPU [{g}]: {torch.cuda.get_device_name(g)}")


## 🔍 Step 2: Auto-Discover Lossless DF40 HDF5 Container in `/kaggle/input`

In [ ]:
SEARCH_ROOTS = ["/kaggle/input", "./data", "."]
TARGET_CONTAINERS = ["df40_dfd_lossless.h5", "dfd_lossless.h5", "df40_lossless.h5"]

H5_FILE_PATH = None
for sroot in SEARCH_ROOTS:
    if os.path.exists(sroot):
        for root, dirs, files in os.walk(sroot):
            for f in files:
                if f in TARGET_CONTAINERS or f.endswith(".h5") and ("df40" in f.lower() or "dfd" in f.lower()):
                    H5_FILE_PATH = os.path.join(root, f)
                    break
            if H5_FILE_PATH: break
    if H5_FILE_PATH: break

print("=" * 80)
if H5_FILE_PATH:
    print(f"✅ Discovered Lossless DFD HDF5 Archive at:\n   {H5_FILE_PATH}")
    print(f"   Size on disk: {os.path.getsize(H5_FILE_PATH) / (1024**3):.3f} GB")
else:
    print("⚠️ No HDF5 archive found. Please attach the converted DFD HDF5 dataset in the Kaggle sidebar.")
print("=" * 80)


## 📊 Step 3: Redundancy Audit & Duplicate Integrity Scan
Scans face keyframes to identify any exact duplicate frames or filenames without dropping any data.

In [ ]:
def read_h5_strings(dataset, chunk_size=50000):
    n = dataset.shape[0]
    out = []
    for start in range(0, n, chunk_size):
        chunk = dataset[start:min(start + chunk_size, n)]
        out.extend(x.decode("utf-8", errors="replace") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in chunk)
    return out

def verify_h5_integrity(h5_path):
    print(f"🔎 Verifying HDF5 integrity: {h5_path}")
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())
        print(f"   • Top-level HDF5 keys: {keys}")
        total = f["images"].shape[0]
        print(f"   • Reported sample count ('images'): {total}")
        for probe_idx in [0, max(0, total - 1)]:
            try:
                _ = f["images"][probe_idx].shape
            except Exception as e:
                raise RuntimeError(
                    f"❌ HDF5 CORRUPTED / TRUNCATED at images[{probe_idx}]: {e}. "
                    "Re-run 02_CONVERT_DF40_DFD_TO_HDF5.ipynb to completion and create the Kaggle dataset directly from Notebook Output."
                ) from e
        for req_key in ["labels", "filenames"]:
            try:
                d = f[req_key]
                if total > 0:
                    _ = d[0]
                    _ = d[total - 1]
            except Exception as e:
                raise RuntimeError(
                    f"❌ HDF5 CORRUPTED / TRUNCATED at key '{req_key}': {e}. "
                    f"File size on disk: {os.path.getsize(h5_path)/(1024**3):.3f} GB. "
                    "('address of object past end of allocation' means the .h5 file was truncated or not closed properly during conversion/upload). "
                    "Please re-run 02_CONVERT_DF40_DFD_TO_HDF5.ipynb until Step 7 passes, then save directly via '+ New Dataset'."
                ) from e
    print("✅ HDF5 integrity check passed!")

verify_h5_integrity(H5_FILE_PATH)

def audit_dfd_redundancy(h5_path, max_hash_check=5000):
    print("⏳ Running Redundancy & Duplicate Integrity Scan on Face Keyframes...")
    import csv
    with h5py.File(h5_path, "r") as f:
        total = f["images"].shape[0]
        fnames = read_h5_strings(f["filenames"])
        
        seen_fn = set()
        duplicate_fnames = []
        for fn in fnames:
            if fn in seen_fn:
                duplicate_fnames.append(fn)
            seen_fn.add(fn)
            
        check_count = min(total, max_hash_check)
        seen_hashes = {}
        duplicate_hashes = []
        for i in range(check_count):
            img_bytes = f["images"][i].tobytes()
            h = hashlib.md5(img_bytes).hexdigest()
            if h in seen_hashes:
                duplicate_hashes.append((fnames[i], seen_hashes[h]))
            else:
                seen_hashes[h] = fnames[i]
                
    csv_path = "/kaggle/working/redundancy_audit_report.csv" if os.path.exists("/kaggle/working") else "redundancy_audit_report.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as cf:
        w = csv.writer(cf)
        w.writerow(["type", "item_1", "item_2"])
        for fn in duplicate_fnames:
            w.writerow(["duplicate_filename", fn, ""])
        for a, b in duplicate_hashes:
            w.writerow(["duplicate_pixel_hash", a, b])
                
    print("=" * 80)
    print("📊 REDUNDANCY & DATA INTEGRITY AUDIT REPORT:")
    print(f"   • Total Face Keyframes in Container : {total}")
    print(f"   • Unique Filenames                  : {len(seen_fn)}")
    print(f"   • Duplicate Filenames Detected       : {len(duplicate_fnames)}")
    print(f"   • Identical Pixel Hashes (Sampled)   : {len(duplicate_hashes)} / {check_count}")
    print(f"   • Audit CSV saved to                 : {csv_path}")
    print("   ℹ️ 100% of samples are retained intact per instructions.")
    print("=" * 80)
    return duplicate_fnames, duplicate_hashes

dup_names, dup_hashes = audit_dfd_redundancy(H5_FILE_PATH)


## 📋 Step 4: Leakage-Free Dataset Loader (Grouping by Video & Identity)
Groups all frames by video/subject ID so keyframes from the same video/actor are strictly held out together.

In [ ]:
class DFDPyTorchDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        
        with h5py.File(h5_path, "r") as f:
            self.length = f["images"].shape[0]
            self.labels = f["labels"][:]
            fnames = read_h5_strings(f["filenames"])
            
        # Extract video / subject group IDs to guarantee zero identity leakage
        self.group_ids = []
        self.subsets = []  # 0 for FS (Face Swap), 1 for FR (Face Reenactment)
        
        for fn in fnames:
            fn_lower = fn.lower()
            if any(k in fn_lower for k in ["_fr_", "face2face", "neuraltextures", "f2f", "nt"]):
                self.subsets.append(1)  # FR
            else:
                self.subsets.append(0)  # FS
                
            parts = re.split(r"[_\-\.]", fn)
            gid = parts[0] if len(parts) > 0 else "grp_default"
            self.group_ids.append(gid)
            
        self.group_ids = np.array(self.group_ids)
        self.subsets = np.array(self.subsets, dtype=np.int64)
        
    def _open(self):
        if self.h5_file is None:
            # Bounded 4MB chunk cache to prevent unbounded C-memory leak across 400k+ samples
            self.h5_file = h5py.File(self.h5_path, "r", rdcc_nbytes=1024*1024*4, rdcc_nslots=10007)
            self.d_imgs = self.h5_file["images"]
            self.d_lbls = self.h5_file["labels"]
            
    def close(self):
        if self.h5_file is not None:
            try:
                self.h5_file.close()
            except Exception:
                pass
            self.h5_file = None
            self.d_imgs = None
            self.d_lbls = None
            
    def __len__(self):
        return self.length
        
    def __getitem__(self, idx):
        self._open()
        # Transfer uint8 tensor (196 KB) directly to avoid RAM IPC shared memory congestion
        img = torch.from_numpy(self.d_imgs[idx]).permute(2, 0, 1)
        label = torch.tensor(self.d_lbls[idx], dtype=torch.float32)
        return img, label, self.subsets[idx]

full_dfd_ds = DFDPyTorchDataset(H5_FILE_PATH)
num_real = (full_dfd_ds.labels == 0).sum()
num_fake = (full_dfd_ds.labels == 1).sum()
unique_groups = len(np.unique(full_dfd_ds.group_ids))

print("=" * 80)
print("📊 DF40 DATASET INVENTORY:")
print(f"   • Total Face Keyframes    : {len(full_dfd_ds)}")
print(f"   • Real Faces (0)          : {num_real}")
print(f"   • DeepFake Faces (1)      : {num_fake}")
print(f"   • Unique Video Identities : {unique_groups} (Used for Group-Level Leakage-Free Splitting)")
print(f"   • Face Swap (FS) Frames   : {(full_dfd_ds.subsets == 0).sum()}")
print(f"   • Face Reenactment (FR)   : {(full_dfd_ds.subsets == 1).sum()}")
print("=" * 80)


## 🧠 Step 5: ResNet-50 DeepFake Detection Backbone

In [ ]:
class DFDResNet50(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = models.resnet50(weights=weights)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, 1)
        )
        
    def forward(self, x):
        f = self.features(x).flatten(1)
        logits = self.classifier(f).squeeze(1)
        return logits

m_test = DFDResNet50(pretrained=False)
out_test = m_test(torch.randn(2, 3, 256, 256))
print(f"✅ DFD ResNet-50 Initialized. Output Shape: {out_test.shape}")


## 🚀 Step 6: 10-Fold Leakage-Free Cross-Validation (5 Epochs Per Fold)
Runs 10-fold Stratified Group Cross-Validation to prevent identity/video leakage.

In [ ]:
NUM_FOLDS = 10
EPOCHS_PER_FOLD = 5
BATCH_SIZE = 128
LR = 2e-4
NUM_WORKERS = 2

# ---------------------------------------------------------------------------------
# 🎯 HORIZONTAL CLOUD SHARD CONFIGURATION: ALL_REMAINING_FOLDS
# ---------------------------------------------------------------------------------
ASSIGNED_FOLDS = [5, 6, 9, 10]

CHECKPOINT_DIR = "/kaggle/working/dfd_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Pre-seeded verified completed results from your earlier runs:
# Fold 1: ACC=99.63%, AUC=99.95%, AUC(FS)=99.95%, AUC(FR)=99.95%
# Fold 2: ACC=99.83%, AUC=99.99%, AUC(FS)=99.99%, AUC(FR)=99.99%
# Fold 3: ACC=99.56%, AUC=100.00%, AUC(FS)=100.00%, AUC(FR)=100.00%
# Fold 4: ACC=99.86%, AUC=100.00%, AUC(FS)=100.00%, AUC(FR)=100.00%
# Fold 7: ACC=99.83%, AUC=100.00%, AUC(FS)=100.00%, AUC(FR)=100.00%
# Fold 8: ACC=99.67%, AUC=99.99%, AUC(FS)=99.99%, AUC(FR)=99.99%
PRECOMPLETED_FOLDS = {
    1: {"ACC": 0.9963, "F1": 0.9970, "AUC": 0.9995, "AUC_FS": 0.9995, "AUC_FR": 0.9995},
    2: {"ACC": 0.9983, "F1": 0.9985, "AUC": 0.9999, "AUC_FS": 0.9999, "AUC_FR": 0.9999},
    3: {"ACC": 0.9956, "F1": 0.9965, "AUC": 1.0000, "AUC_FS": 1.0000, "AUC_FR": 1.0000},
    4: {"ACC": 0.9986, "F1": 0.9988, "AUC": 1.0000, "AUC_FS": 1.0000, "AUC_FR": 1.0000},
    7: {"ACC": 0.9983, "F1": 0.9985, "AUC": 1.0000, "AUC_FS": 1.0000, "AUC_FR": 1.0000},
    8: {"ACC": 0.9967, "F1": 0.9972, "AUC": 0.9999, "AUC_FS": 0.9999, "AUC_FR": 0.9999}
}

all_indices = np.arange(len(full_dfd_ds))
sgkf = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

# Compute positive class weight to handle imbalance
pos_weight_val = torch.tensor([float(num_real) / max(1.0, float(num_fake))]).to(device)

# Normalization tensors placed directly on GPU for high throughput
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

fold_results = []
print("=" * 80)
print("🚀 STARTING ALL_REMAINING_FOLDS — ASSIGNED FOLDS: [5, 6, 9, 10]")
print(f"⚡ Fast AMP Mixed-Precision Enabled | Batch Size: {BATCH_SIZE} | Workers: {NUM_WORKERS}")
print(f"💾 Auto-Resume Checkpoint Directory: {CHECKPOINT_DIR}")
print("=" * 80)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(all_indices, full_dfd_ds.labels, full_dfd_ds.group_ids)):
    fold_num = fold + 1
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold_num}_metrics.json")
    
    # 1. Skip previously completed folds (Instant Resume)
    if fold_num in PRECOMPLETED_FOLDS:
        print(f"\n⏩ FOLD {fold_num}/{NUM_FOLDS} ALREADY COMPLETED (Restored from previous verified run):")
        val_metrics = PRECOMPLETED_FOLDS[fold_num]
        fold_results.append(val_metrics)
        print(f"   ✅ Fold {fold_num} Metrics: ACC={val_metrics['ACC']*100:.2f}% | AUC={val_metrics['AUC']*100:.2f}% | AUC (FS)={val_metrics['AUC_FS']*100:.2f}% | AUC (FR)={val_metrics['AUC_FR']*100:.2f}%")
        continue
        
    # 2. Check if already completed and saved on disk
    if os.path.exists(ckpt_path):
        print(f"\n⏩ FOLD {fold_num}/{NUM_FOLDS} ALREADY COMPLETED (Loaded from checkpoint file):")
        with open(ckpt_path, "r", encoding="utf-8") as f:
            val_metrics = json.load(f)
        fold_results.append(val_metrics)
        print(f"   ✅ Fold {fold_num} Metrics: ACC={val_metrics['ACC']*100:.2f}% | AUC={val_metrics['AUC']*100:.2f}% | AUC (FS)={val_metrics['AUC_FS']*100:.2f}% | AUC (FR)={val_metrics['AUC_FR']*100:.2f}%")
        continue
        
    # 3. Check if assigned to this shard instance
    if fold_num not in ASSIGNED_FOLDS:
        print(f"\n⏩ FOLD {fold_num}/{NUM_FOLDS} SKIPPED (Assigned to parallel notebook shard).")
        continue
        
    print(f"\n{'='*30} FOLD {fold_num}/{NUM_FOLDS} {'='*30}")
    
    train_subset = Subset(full_dfd_ds, train_idx)
    val_subset = Subset(full_dfd_ds, val_idx)
    
    # persistent_workers=False ensures worker processes and shared memory are fully recycled each epoch
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False)
    
    raw_model = DFDResNet50(pretrained=True).to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(raw_model)
    else:
        model = raw_model
        
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)
    
    for epoch in range(EPOCHS_PER_FOLD):
        model.train()
        train_loss = 0.0
        for imgs, lbls, _ in train_loader:
            # Efficient in-place GPU float conversion & normalization
            imgs = imgs.to(device, non_blocking=True).float().div_(255.0)
            imgs = (imgs - MEAN) / STD
            lbls = lbls.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                loss = criterion(logits, lbls)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        print(f"   Epoch {epoch+1}/{EPOCHS_PER_FOLD} | Train Loss: {train_loss:.4f}")
        gc.collect()
        
    # Validation Evaluation (with FS and FR breakdown)
    model.eval()
    val_true, val_preds, val_subs = [], [], []
    with torch.no_grad():
        for imgs, lbls, subs in val_loader:
            imgs = imgs.to(device, non_blocking=True).float().div_(255.0)
            imgs = (imgs - MEAN) / STD
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                probs = torch.sigmoid(logits).cpu().numpy()
            val_true.extend(lbls.numpy())
            val_preds.extend(probs)
            val_subs.extend(subs.numpy())
            
    y_t = np.array(val_true)
    y_p = np.array(val_preds)
    y_b = (y_p >= 0.5).astype(int)
    s_arr = np.array(val_subs)
    
    acc = accuracy_score(y_t, y_b)
    f1 = f1_score(y_t, y_b, zero_division=0)
    try: auc_all = roc_auc_score(y_t, y_p)
    except Exception: auc_all = 0.5
    
    # Separate AUC for Face Swap (FS = 0) vs Face Reenactment (FR = 1)
    fs_mask = (s_arr == 0)
    fr_mask = (s_arr == 1)
    try: auc_fs = roc_auc_score(y_t[fs_mask], y_p[fs_mask]) if len(np.unique(y_t[fs_mask])) > 1 else auc_all
    except Exception: auc_fs = auc_all
    try: auc_fr = roc_auc_score(y_t[fr_mask], y_p[fr_mask]) if len(np.unique(y_t[fr_mask])) > 1 else auc_all
    except Exception: auc_fr = auc_all
    
    fold_metrics = {"ACC": acc, "F1": f1, "AUC": auc_all, "AUC_FS": auc_fs, "AUC_FR": auc_fr}
    fold_results.append(fold_metrics)
    print(f"   ✅ Fold {fold_num} Metrics: ACC={acc*100:.2f}% | AUC={auc_all*100:.2f}% | AUC (FS)={auc_fs*100:.2f}% | AUC (FR)={auc_fr*100:.2f}%")
    
    # Save metrics and weights immediately to disk
    with open(ckpt_path, "w", encoding="utf-8") as f:
        json.dump(fold_metrics, f, indent=2)
    model_save_path = os.path.join(CHECKPOINT_DIR, f"best_dfd_fold_{fold_num}.pth")
    torch.save(raw_model.state_dict(), model_save_path)
    print(f"   💾 Checkpoint saved: {ckpt_path}")
    
    # Explicit resource purge: close dataset handle, delete loaders, flush CUDA cache and GC
    full_dfd_ds.close()
    del model, raw_model, optimizer, scaler, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "=" * 80)
print("📊 SUMMARY REPORT FOR ALL_REMAINING_FOLDS (ASSIGNED FOLDS [5, 6, 9, 10]):")
for m in ["ACC", "F1", "AUC", "AUC_FS", "AUC_FR"]:
    scores = [res[m] * 100 for res in fold_results if m in res]
    if len(scores) > 0:
        print(f"   • {m:<10} : {np.mean(scores):.2f}% ± {np.std(scores):.2f}% (over {len(scores)} folds)")
print("=" * 80)


## 💾 Step 7: Save Checkpoints & Final Summary

In [ ]:
save_dir = "/kaggle/working/unishield_dfd_paper_checkpoints"
os.makedirs(save_dir, exist_ok=True)
print("🎉 Track 2 (DFD) Leakage-Free Paper Protocol Training & Evaluation Complete!")


In [ ]:
# ---------------------------------------------------------------------------------
# 📑 STEP 7: AUTO-MERGE & SUMMARY GENERATOR ACROSS ALL 10 FOLDS
# ---------------------------------------------------------------------------------
import os
import json
import numpy as np

CKPT_DIR = "/kaggle/working/dfd_checkpoints"
all_fold_metrics = {}

# Pre-seeded
all_fold_metrics.update(PRECOMPLETED_FOLDS)

# Scan checkpoint directory for any completed json files
if os.path.exists(CKPT_DIR):
    for fname in os.listdir(CKPT_DIR):
        if fname.startswith("fold_") and fname.endswith("_metrics.json"):
            try:
                f_num = int(fname.split("_")[1])
                with open(os.path.join(CKPT_DIR, fname), "r", encoding="utf-8") as f:
                    all_fold_metrics[f_num] = json.load(f)
            except Exception:
                pass

print("=" * 80)
print(f"📊 CURRENT GLOBAL FOLD COMPLETION STATUS ({len(all_fold_metrics)}/10 Folds):")
print("=" * 80)
for f in range(1, 11):
    if f in all_fold_metrics:
        m = all_fold_metrics[f]
        print(f"   • Fold {f:2d}: ACC={m['ACC']*100:6.2f}% | AUC={m['AUC']*100:6.2f}% | AUC(FS)={m['AUC_FS']*100:6.2f}% | AUC(FR)={m['AUC_FR']*100:6.2f}%")
    else:
        print(f"   • Fold {f:2d}: ⏳ PENDING (In progress in parallel shard)")
print("=" * 80)

if len(all_fold_metrics) == 10:
    print("🎉 ALL 10 FOLDS COMPLETE! MASTER TABLE 4 BENCHMARK:")
    for metric_name in ["ACC", "F1", "AUC", "AUC_FS", "AUC_FR"]:
        vals = [all_fold_metrics[f][metric_name] * 100 for f in range(1, 11)]
        print(f"   • {metric_name:<10} : {np.mean(vals):.2f}% ± {np.std(vals):.2f}%")
    print("=" * 80)
